# 📘 03 – Data Preprocessing

## 🔍 Въведение
Този notebook е посветен на обработката на данните за задачата *Hierarchical Demand Forecasting – M5 Walmart*. Преди да изградим модел за прогнозиране с помощта на невронна мрежа (MLP), е необходимо да подготвим данните така, че те да бъдат подходящи за машинно обучение. Суровият dataset в своя оригинален вид не е директно приложим – той съдържа времеви редове във wide формат, множество категориални променливи и липсващи контекстуални характеристики.

Обработката на данните (Data Preprocessing) е критичен етап, тъй като тя гарантира:
- последователност и валидност на данните,
- избягване на data leakage,
- създаване на смислени входни характеристики (features),
- подготовка на вход, съвместим с архитектурата на MLP модела.

---

## 🎯 Цел на този notebook
Основната цел е да изградим **производствено готов pipeline** за подготовка на данните, който:
✅ Поддържа времевата структура на данните  
✅ Извършва feature engineering (lag, rolling, calendar features)  
✅ Подготвя входни данни за невронната мрежа  
✅ Скалира числовите стойности  
✅ Кодира категориалните данни  
✅ Създава train/validation/test splits във времеви ред  
✅ Запазва всички preprocessing обекти (encoder-и, scaler-и)

---

## 🛠️ Подход
Заради големия обем (~59M реда) следваме **поетапна стратегия за обработка**:

| Етап | Обем данни | Цел |
|------|------------|-----|
| **Sample Processing** | ~100 продукта | Тест на pipeline и feature engineering |
| **Chunk Processing** | 50–100K реда | Ефективна обработка без RAM проблеми |
| **Full Dataset** | 30K продукта | Финално обучение на модел (опционално тук) |

---

## ✅ План на notebook-а
Структурата на този notebook е следната:

| Секция | Описание |
|--------|-----------|
| **0. Setup** | Импортиране на библиотеки, константи, пътища, seed |
| **1. Load Meta Data** | Зареждане на calendar и prices + schema за sales |
| **2. Helper Functions** | Memory optimization + lag & rolling генератори |
| **3. Processing Strategy** | Обяснение на workflow-а за обработка |
| **4. Sample Transformation** | Преобразуване на sample dataset |
| **5. Full Processing (optional)** | Chunk обработка за целия dataset |
| **6. Time-based Split** | Train/Validation/Test с времева зависимост |
| **7. Scaling/Encoding** | StandardScaler + OneHot/Label Encoding |
| **8. Save Pipeline Objects** | Запис на preprocess артефакти |
| **9. Sanity Checks** | Проверки за нови features |

---

След този notebook данните ще бъдат готови за:

✅ Обучение на първи baseline модел  
✅ Зареждане в PyTorch/Keras MLP  
✅ Използване в Flask приложението  

---

## 0. Setup



In [2]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

# Проектови пътища (от data/ към root)
PROJ_ROOT = Path.cwd().resolve().parent
DATA_RAW = PROJ_ROOT / "data" 
DATA_INTERIM = PROJ_ROOT / "data" 
DATA_PROCESSED = PROJ_ROOT / "data" 
MODELS_DIR = PROJ_ROOT / "models"
REPORTS = PROJ_ROOT / "reports"

for path in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, MODELS_DIR, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

# Seed-ове за възпроизводимост
GLOBAL_SEED = 42

def set_all_seeds(seed: int = GLOBAL_SEED):
    np.random.seed(seed)
    random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

set_all_seeds()

# Pandas дисплей настройки
pd.options.display.max_rows = 200
pd.options.display.float_format = "{:,.4f}".format



## 1. Load Meta Data



In [3]:
# Calendar (date features + SNAP)
cal_usecols = [
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "d",
]
calendar = pd.read_csv(
    DATA_RAW / "calendar.csv",
    usecols=cal_usecols,
    parse_dates=["date"],
)

# Sell prices (schema/head за sanity check)
sp_usecols = ["store_id", "item_id", "wm_yr_wk", "sell_price"]
sell_prices_head = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=sp_usecols,
    nrows=50,
)
sell_prices_types = {c: "string" for c in ["store_id", "item_id"]}
sell_prices_types["wm_yr_wk"] = "int32"
sell_prices_types["sell_price"] = "float32"

# Sales metadata (ID нива без d_* колоните)
meta_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
sales_meta = pd.read_csv(
    DATA_RAW / "sales_train_evaluation.csv",
    usecols=meta_cols,
    dtype="string",
)

schema_info = {
    "calendar_cols": list(calendar.columns),
    "sell_prices_cols": sp_usecols,
    "sales_meta_cols": meta_cols,
    "n_items": sales_meta["item_id"].nunique(),
    "n_stores": sales_meta["store_id"].nunique(),
}

print(schema_info)



{'calendar_cols': ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI'], 'sell_prices_cols': ['store_id', 'item_id', 'wm_yr_wk', 'sell_price'], 'sales_meta_cols': ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], 'n_items': 3049, 'n_stores': 10}


## 2. Helper Functions



In [4]:
def reduce_memory_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if str(col_type).startswith("int"):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif str(col_type).startswith("float"):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif col_type == "object":
            # малки card. → category
            if df[col].nunique(dropna=False) / len(df[col]) < 0.5:
                df[col] = df[col].astype("category")
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose and start_mem:
        saved = 100 * (start_mem - end_mem) / start_mem
        print(f"Mem: {start_mem:0.2f} → {end_mem:0.2f} MB ({saved:0.1f}% saved)")
    return df


def create_lag_features(df: pd.DataFrame, target_col: str, lags=(7, 14, 28)):
    for lag in lags:
        df[f"{target_col}_lag{lag}"] = (
            df.groupby(["item_id", "store_id"], observed=True)[target_col]
            .shift(lag)
        )
    return df


def create_rolling_features(df: pd.DataFrame, target_col: str, windows=(7, 28)):
    for window in windows:
        grp = df.groupby(["item_id", "store_id"], observed=True)[target_col]
        df[f"{target_col}_rmean{window}"] = grp.shift(1).rolling(window).mean()
        df[f"{target_col}_rstd{window}"] = grp.shift(1).rolling(window).std()
    return df


def encode_cyclic_features(df: pd.DataFrame, col: str, max_val: int):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / max_val)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / max_val)
    return df


def time_based_split(df: pd.DataFrame, date_col: str, val_start: str, test_start: str):
    train_df = df[df[date_col] < val_start]
    val_df = df[(df[date_col] >= val_start) & (df[date_col] < test_start)]
    test_df = df[df[date_col] >= test_start]
    return train_df, val_df, test_df



## 3. Processing Strategy

**Workflow:**
1. Извличаме малък sample (1 store × 1 category), за да валидираме стъпките.
2. Преобразуваме wide `sales_train_evaluation.csv` към long формат.
3. Merge-ваме календар и цени, за да добавим времеви и ценови контекст.
4. Генерираме lag/rolling/cyclic features.
5. Чистим NaN редове и записваме sample артефакт.



## 4. Transform sample (Proof-of-concept)

**Цел:** Демонстрация върху малък dataset.

**Какво правим:**

- Вземаме само **1 store + 1 category**
- Merge с calendar и prices
- Създаваме lag и rolling features
- Проверяваме дали няма проблеми с NaN или размери



In [5]:
# 1) Вземаме само 1 store + 1 category
sample_store = "CA_1"
sample_cat = "HOBBIES"

print(f"Избиране на sample: {sample_store} + {sample_cat}")

meta_sample = sales_meta.query(
    "store_id == @sample_store and cat_id == @sample_cat"
).copy()
sample_ids = set(meta_sample["id"])

print(f"Намерени {len(sample_ids)} продукта")

# Зареждаме wide sales данни само за sample ID-тата
header_cols = pd.read_csv(
    DATA_RAW / "sales_train_evaluation.csv",
    nrows=0,
).columns.tolist()
value_cols = [c for c in header_cols if c.startswith("d_")]
usecols = list(dict.fromkeys(meta_cols + value_cols))

# Chunk-by-chunk зареждане за ефективност
filtered_chunks = []
for chunk in pd.read_csv(
    DATA_RAW / "sales_train_evaluation.csv",
    usecols=usecols,
    chunksize=100_000,
):
    mask = chunk["id"].isin(sample_ids)
    if mask.any():
        filtered_chunks.append(chunk.loc[mask].copy())

if not filtered_chunks:
    raise ValueError("Няма намерени редове за избрания sample.")

wide = pd.concat(filtered_chunks, ignore_index=True)

# Преобразуване от wide към long формат
long = wide.melt(
    id_vars=meta_cols,
    value_vars=value_cols,
    var_name="d",
    value_name="sales",
)
long["sales"] = long["sales"].astype("int16")

print(f"Long формат: {long.shape}")

# 2) Merge с calendar и prices
calendar_cols = [
    "d",
    "date",
    "wm_yr_wk",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]
calendar_subset = calendar[calendar_cols].copy()

# Зареждане на sell prices
sell_prices = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
)
sell_prices["sell_price"] = sell_prices["sell_price"].astype("float32")

# Merge операции
sample_df = long.merge(calendar_subset, on="d", how="left")
sample_df = sample_df.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
)

print(f"След merge с calendar и prices: {sample_df.shape}")

# 3) Създаваме lag и rolling features
sample_df = sample_df.sort_values(["item_id", "store_id", "date"])

# Lag features (7, 14, 28 дни назад)
sample_df = create_lag_features(sample_df, target_col="sales", lags=(7, 14, 28))

# Rolling features (средно и стандартно отклонение)
sample_df = create_rolling_features(sample_df, target_col="sales", windows=(7, 28))

print(f"След feature engineering: {sample_df.shape}")

# 4) Проверяваме дали няма проблеми с NaN или размери

# Премахваме редовете без валидни lag/rolling (първите 28 дни)
min_window = 28
min_date = sample_df["date"].min() + pd.Timedelta(days=min_window)
sample_df = sample_df[sample_df["date"] >= min_date].copy()

print(f"\nПроверки за NaN:")
print(f"Общо редове: {len(sample_df)}")
print(f"NaN в lag features: {sample_df[['sales_lag7', 'sales_lag14', 'sales_lag28']].isna().sum().sum()}")
print(f"NaN в rolling features: {sample_df[['sales_rmean7', 'sales_rmean28']].isna().sum().sum()}")

# Проверка за критични колони
assert sample_df[["sales_lag7", "sales_rmean7"]].isna().sum().sum() == 0, \
    "Има NaN в lag/rolling features!"

print(f"\nНай-често липсващи стойности:")
nan_summary = sample_df.isna().mean().sort_values(ascending=False)
print(nan_summary[nan_summary > 0].head(10))

print(f"\n✅ Sample обработка завършена успешно!")
print(f"Финален размер: {sample_df.shape}")



Избиране на sample: CA_1 + HOBBIES
Намерени 565 продукта
Long формат: (1096665, 8)
След merge с calendar и prices: (1096665, 19)
След feature engineering: (1096665, 26)

Проверки за NaN:
Общо редове: 1080845
NaN в lag features: 0
NaN в rolling features: 0

Най-често липсващи стойности:
event_name_1   0.9190
event_type_1   0.9190
sell_price     0.1835
dtype: float64

✅ Sample обработка завършена успешно!
Финален размер: (1080845, 26)


## **5. Full processing (optional – за мощна машина)**

**Цел:** Подгответе code, но може да се пропусне локално.

**Какво включва:**

- Chunk обработка (100K реда на парче)
- Save като .parquet за оптимална скорост


In [ ]:
RUN_FULL_PROCESSING = False  

if not RUN_FULL_PROCESSING:
    print("⏭️ Full processing е пропуснато (RUN_FULL_PROCESSING = False)")
    print("   Можете да смените на True, ако искате да обработите целия dataset")
else:
    print("🚀 Стартиране на full processing...")
    print(f"Очакван обем: ~{len(sales_meta)} продукта × ~1969 дни = ~59M реда")
    
    # Подготовка на calendar и prices (веднъж за всички chunks)
    calendar_cols = [
        "d",
        "date",
        "wm_yr_wk",
        "wday",
        "month",
        "year",
        "event_name_1",
        "event_type_1",
        "snap_CA",
        "snap_TX",
        "snap_WI",
    ]
    calendar_subset = calendar[calendar_cols].copy()
    
    sell_prices = pd.read_csv(
        DATA_RAW / "sell_prices.csv",
        usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    )
    sell_prices["sell_price"] = sell_prices["sell_price"].astype("float32")
    
    # Подготовка на value колони
    header_cols = pd.read_csv(
        DATA_RAW / "sales_train_evaluation.csv",
        nrows=0,
    ).columns.tolist()
    value_cols = [c for c in header_cols if c.startswith("d_")]
    usecols = list(dict.fromkeys(meta_cols + value_cols))
    
    # Параметри за chunk processing
    CHUNK_SIZE = 100_000  # редове на chunk
    OUTPUT_DIR = DATA_INTERIM / "processed_chunks"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print(f"\n📦 Chunk размер: {CHUNK_SIZE:,} реда")
    print(f"💾 Output директория: {OUTPUT_DIR}")
    
    chunk_num = 0
    total_rows = 0
    
    # Chunk-by-chunk обработка
    for chunk in pd.read_csv(
        DATA_RAW / "sales_train_evaluation.csv",
        usecols=usecols,
        chunksize=CHUNK_SIZE,
    ):
        chunk_num += 1
        print(f"\n📊 Обработка на chunk {chunk_num} ({len(chunk):,} реда)...")
        
        # 1) Wide → Long
        long_chunk = chunk.melt(
            id_vars=meta_cols,
            value_vars=value_cols,
            var_name="d",
            value_name="sales",
        )
        long_chunk["sales"] = long_chunk["sales"].astype("int16")
        
        # 2) Merge с calendar и prices
        df_chunk = long_chunk.merge(calendar_subset, on="d", how="left")
        df_chunk = df_chunk.merge(
            sell_prices,
            on=["store_id", "item_id", "wm_yr_wk"],
            how="left",
        )
        
        # 3) Сортиране за feature engineering
        df_chunk = df_chunk.sort_values(["item_id", "store_id", "date"])
        
        # 4) Feature engineering
        df_chunk = create_lag_features(df_chunk, target_col="sales", lags=(7, 14, 28))
        df_chunk = create_rolling_features(df_chunk, target_col="sales", windows=(7, 28))
        
        # 5) Премахване на редове без валидни lag/rolling
        min_window = 28
        min_date = df_chunk["date"].min() + pd.Timedelta(days=min_window)
        df_chunk = df_chunk[df_chunk["date"] >= min_date].copy()
        
        # 6) Memory optimization
        df_chunk = reduce_memory_usage(df_chunk, verbose=False)
        
        # 7) Запис като parquet
        output_file = OUTPUT_DIR / f"chunk_{chunk_num:04d}.parquet"
        df_chunk.to_parquet(output_file, index=False, engine="pyarrow")
        
        total_rows += len(df_chunk)
        print(f"   ✅ Записано: {output_file.name} ({len(df_chunk):,} реда)")
    
    print(f"\n✅ Full processing завършен!")
    print(f"📈 Общо обработени редове: {total_rows:,}")
    print(f"📁 Записани файлове: {chunk_num} chunks в {OUTPUT_DIR}")
    
    # Информация за комбиниране на chunks (ако е необходимо)
    print(f"\n💡 За да комбинирате chunks в един файл:")
    print(f"   chunks = [pd.read_parquet(f) for f in sorted({OUTPUT_DIR}.glob('chunk_*.parquet'))]")
    print(f"   full_df = pd.concat(chunks, ignore_index=True)")


⏭️ Full processing е пропуснато (RUN_FULL_PROCESSING = False)
   Можете да смените на True, ако искате да обработите целия dataset


## **6. Create X,y splits (time-aware)**

**Цел:** Преобразуваме таблицата към supervised learning формат.

**Какво правим:**

- y = future sales (прогноза за 1 ден напред)
- Избираме train/valid/test спрямо времето:
  - Train: ~70% от началните дати
  - Valid: ~15% (следващи дати)
  - Test: ~15% (най-новите дати)
- Създаваме X (features) и y (target) за всеки split


In [ ]:

try:
    sample_df  # Проверка дали sample_df съществува
    print("✅ Използване на sample_df от секция 4")
except NameError:
    # Ако няма sample_df, зареждаме от parquet
    sample_path = DATA_INTERIM / "sample_features.parquet"
    if sample_path.exists():
        sample_df = pd.read_parquet(sample_path)
        print(f"✅ Заредени sample данни от {sample_path}")
    else:
        raise ValueError("Няма налични данни! Изпълнете секция 4 първо.")

print(f"\nНачален размер: {sample_df.shape}")
print(f"Дата обхват: {sample_df['date'].min()} до {sample_df['date'].max()}")

# 1) Създаване на y (target) - sales 1 ден напред
sample_df = sample_df.sort_values(["item_id", "store_id", "date"])
sample_df["y"] = (
    sample_df.groupby(["item_id", "store_id"], observed=True)["sales"]
    .shift(-1)  # Следващия ден
)

# Премахване на редове без валиден target (последният ден за всеки продукт)
sample_df = sample_df.dropna(subset=["y"]).copy()

print(f"\nСлед добавяне на target (y): {sample_df.shape}")
print(f"Премахнати редове без target: {(sample_df['y'].isna().sum())}")

date_col = "date"
dates = sample_df[date_col].sort_values()

# Пропорционално разделение: 70% train, 15% valid, 15% test
total_days = len(dates.unique())
train_end_idx = int(0.7 * total_days)
val_end_idx = int(0.85 * total_days)

train_end_date = dates.unique()[train_end_idx]
val_end_date = dates.unique()[val_end_idx]

print(f"\n📅 Time-based split:")
print(f"   Train: до {train_end_date} (~70%)")
print(f"   Valid: {train_end_date} до {val_end_date} (~15%)")
print(f"   Test:  от {val_end_date} (~15%)")

# Създаване на splits
train_df = sample_df[sample_df[date_col] < train_end_date].copy()
val_df = sample_df[
    (sample_df[date_col] >= train_end_date) & (sample_df[date_col] < val_end_date)
].copy()
test_df = sample_df[sample_df[date_col] >= val_end_date].copy()

print(f"\n✅ Splits създадени:")
print(f"   Train: {len(train_df):,} редове ({len(train_df) / len(sample_df) * 100:.1f}%)")
print(f"   Valid: {len(val_df):,} редове ({len(val_df) / len(sample_df) * 100:.1f}%)")
print(f"   Test:  {len(test_df):,} редове ({len(test_df) / len(sample_df) * 100:.1f}%)")

# 3) Разделяне на X (features) и y (target)
# Първо идентифицираме кои колони са features

# Колони, които НЕ са features:
exclude_cols = [
    "id",
    "date",
    "d",
    "sales",  # текущите sales (използваме само lag features)
    "y",  # target
]

# Всички останали колони са features
feature_cols = [c for c in sample_df.columns if c not in exclude_cols]

print(f"\n📊 Features ({len(feature_cols)} колони):")
print(f"   {', '.join(feature_cols[:10])}...")  # Първите 10

# Създаване на X, y за всеки split
X_train = train_df[feature_cols].copy()
y_train = train_df["y"].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df["y"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["y"].copy()

print(f"\n✅ X, y splits готови:")
print(f"   X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"   X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"   X_test:  {X_test.shape}, y_test:  {y_test.shape}")

# Проверки
assert len(X_train) == len(y_train), "Train split размер несъответствие!"
assert len(X_val) == len(y_val), "Validation split размер несъответствие!"
assert len(X_test) == len(y_test), "Test split размер несъответствие!"

print(f"\n✅ Всички проверки преминати успешно!")


✅ Използване на sample_df от секция 4

Начален размер: (1080845, 26)
Дата обхват: 2011-02-26 00:00:00 до 2016-05-22 00:00:00

След добавяне на target (y): (1080280, 27)
Премахнати редове без target: 0

📅 Time-based split:
   Train: до 2014-10-26 00:00:00 (~70%)
   Valid: 2014-10-26 00:00:00 до 2015-08-09 00:00:00 (~15%)
   Test:  от 2015-08-09 00:00:00 (~15%)

✅ Splits създадени:
   Train: 755,970 редове (70.0%)
   Valid: 162,155 редове (15.0%)
   Test:  162,155 редове (15.0%)

📊 Features (22 колони):
   item_id, dept_id, cat_id, store_id, state_id, wm_yr_wk, wday, month, year, event_name_1...

✅ X, y splits готови:
   X_train: (755970, 22), y_train: (755970,)
   X_val:   (162155, 22), y_val:   (162155,)
   X_test:  (162155, 22), y_test:  (162155,)

✅ Всички проверки преминати успешно!


## **7. Scaling & Encoding**

**Цел:** Подготовка за MLP.

**Какво правим:**

- MinMaxScaler за числови данни
- LabelEncoder/Embedding за категориите
- Important: fit scaler само върху train!


In [8]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pickle

# Проверка дали X_train, y_train съществуват
try:
    X_train, y_train, X_val, y_val, X_test, y_test
    print("✅ Използване на X, y splits от секция 6")
except NameError:
    raise ValueError("Няма налични X, y splits! Изпълнете секция 6 първо.")

print(f"\nНачални размери:")
print(f"   X_train: {X_train.shape}")
print(f"   X_val:   {X_val.shape}")
print(f"   X_test:  {X_test.shape}")

# 1) Проверка за NaN стойности преди обработка
print(f"\n🔍 Проверка за NaN стойности преди обработка:")
nan_in_train = X_train.isna().sum()
nan_in_val = X_val.isna().sum()
nan_in_test = X_test.isna().sum()

if nan_in_train.sum() > 0:
    print(f"   ⚠️  NaN в train данните:")
    print(nan_in_train[nan_in_train > 0])
if nan_in_val.sum() > 0:
    print(f"   ⚠️  NaN в val данните:")
    print(nan_in_val[nan_in_val > 0])
if nan_in_test.sum() > 0:
    print(f"   ⚠️  NaN в test данните:")
    print(nan_in_test[nan_in_test > 0])

# 2) Идентификация на числови и категориални колони
numeric_cols = []
categorical_cols = []

for col in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[col]):
        # Пропускаме колони, които вече са NaN или имат малко уникални стойности
        if X_train[col].nunique() > 2:  # Избягваме binary колони като flags
            numeric_cols.append(col)
    else:
        categorical_cols.append(col)

print(f"\n📊 Идентификация на колони:")
print(f"   Числови: {len(numeric_cols)} ({', '.join(numeric_cols[:5])}...)")
print(f"   Категориални: {len(categorical_cols)} ({', '.join(categorical_cols[:5]) if categorical_cols else 'няма'}...)")

# 3) Обработка на NaN стойности в числени колони
print(f"\n🔧 Обработка на NaN стойности...")

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# За числени колони: попълваме NaN с медиана от train данните
for col in numeric_cols:
    if X_train[col].isna().any() or X_val[col].isna().any() or X_test[col].isna().any():
        # Изчисляваме fill value от train данните
        median_val = X_train[col].median()
        if pd.isna(median_val):
            # Ако медианата е NaN (всички стойности са NaN), използваме 0
            fill_val = 0
            print(f"   ⚠️  {col}: всички стойности NaN, попълнено с 0")
        else:
            fill_val = median_val
            print(f"   ⚠️  {col}: попълнено с медиана {fill_val:.4f}")
        
        X_train_scaled[col] = X_train_scaled[col].fillna(fill_val)
        X_val_scaled[col] = X_val_scaled[col].fillna(fill_val)
        X_test_scaled[col] = X_test_scaled[col].fillna(fill_val)

# 4) Scaling на числовите данни
# ⚠️ ВАЖНО: Fit само върху train данните!
print(f"\n🔢 Scaling на числови данни...")

scaler = MinMaxScaler()
# Fit и transform само върху train
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_scaled[numeric_cols])
# Transform на val и test (без fit!)
X_val_scaled[numeric_cols] = scaler.transform(X_val_scaled[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test_scaled[numeric_cols])

print(f"   ✅ MinMaxScaler приложен (fit само върху train)")
print(f"   Обхват на скалираните данни: [{X_train_scaled[numeric_cols].min().min():.3f}, {X_train_scaled[numeric_cols].max().max():.3f}]")

# 5) Encoding на категориалните данни
print(f"\n🏷️  Encoding на категориални данни...")

label_encoders = {}

if categorical_cols:
    for col in categorical_cols:
        le = LabelEncoder()
        
        # Fit само върху train данните
        # Попълваме NaN стойности с "Unknown"
        train_values = X_train[col].fillna("Unknown").astype(str)
        le.fit(train_values)
        
        # Transform на всички splits
        # Попълваме NaN с "Unknown" и обработваме неизвестни категории
        train_transformed = X_train[col].fillna("Unknown").astype(str)
        val_transformed = X_val[col].fillna("Unknown").astype(str)
        test_transformed = X_test[col].fillna("Unknown").astype(str)
        
        # Заменяме неизвестни категории в val/test с "Unknown"
        # (но само ако "Unknown" е в classes - което е така, защото сме го добавили)
        val_transformed = val_transformed.apply(lambda x: x if x in le.classes_ else "Unknown")
        test_transformed = test_transformed.apply(lambda x: x if x in le.classes_ else "Unknown")
        
        X_train_scaled[col] = le.transform(train_transformed)
        X_val_scaled[col] = le.transform(val_transformed)
        X_test_scaled[col] = le.transform(test_transformed)
        
        label_encoders[col] = le
        print(f"   ✅ {col}: {len(le.classes_)} категории")
else:
    print(f"   ℹ️  Няма категориални колони за encoding")

# 6) Запазване на scaler и encoders за бъдеща употреба
preprocessing_objects = {
    "scaler": scaler,
    "label_encoders": label_encoders,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "feature_cols": list(X_train.columns),
}

output_path = MODELS_DIR / "preprocessing_objects.pkl"
with open(output_path, "wb") as f:
    pickle.dump(preprocessing_objects, f)

print(f"\n💾 Preprocessing обекти запазени в: {output_path}")

# 7) Проверки
print(f"\n✅ Проверки:")
print(f"   X_train_scaled: {X_train_scaled.shape}")
print(f"   X_val_scaled:   {X_val_scaled.shape}")
print(f"   X_test_scaled:  {X_test_scaled.shape}")

# Проверка за NaN стойности с детайли
train_nan = X_train_scaled.isna().sum().sum()
val_nan = X_val_scaled.isna().sum().sum()
test_nan = X_test_scaled.isna().sum().sum()

if train_nan > 0:
    print(f"\n   ⚠️  NaN в X_train_scaled ({train_nan}):")
    print(X_train_scaled.isna().sum()[X_train_scaled.isna().sum() > 0])
if val_nan > 0:
    print(f"\n   ⚠️  NaN в X_val_scaled ({val_nan}):")
    print(X_val_scaled.isna().sum()[X_val_scaled.isna().sum() > 0])
if test_nan > 0:
    print(f"\n   ⚠️  NaN в X_test_scaled ({test_nan}):")
    print(X_test_scaled.isna().sum()[X_test_scaled.isna().sum() > 0])

if train_nan == 0 and val_nan == 0 and test_nan == 0:
    print(f"   ✅ Няма NaN стойности")
else:
    # Ако все още има NaN, попълваме ги с 0 за числени колони
    print(f"   🔧 Попълване на оставащи NaN с 0...")
    X_train_scaled = X_train_scaled.fillna(0)
    X_val_scaled = X_val_scaled.fillna(0)
    X_test_scaled = X_test_scaled.fillna(0)
    print(f"   ✅ Всички NaN стойности обработени")

# Проверка за infinity стойности
assert not np.isinf(X_train_scaled.select_dtypes(include=[np.number])).any().any(), \
    "Има infinity стойности в X_train_scaled!"
assert not np.isinf(X_val_scaled.select_dtypes(include=[np.number])).any().any(), \
    "Има infinity стойности в X_val_scaled!"
assert not np.isinf(X_test_scaled.select_dtypes(include=[np.number])).any().any(), \
    "Има infinity стойности в X_test_scaled!"

print(f"   ✅ Няма infinity стойности")
print(f"\n✅ Scaling & Encoding завършено успешно!")


✅ Използване на X, y splits от секция 6

Начални размери:
   X_train: (755970, 22)
   X_val:   (162155, 22)
   X_test:  (162155, 22)

🔍 Проверка за NaN стойности преди обработка:
   ⚠️  NaN в train данните:
event_name_1    696645
event_type_1    696645
sell_price      197585
dtype: int64
   ⚠️  NaN в val данните:
event_name_1    146900
event_type_1    146900
sell_price         774
dtype: int64
   ⚠️  NaN в test данните:
event_name_1    149160
event_type_1    149160
dtype: int64

📊 Идентификация на колони:
   Числови: 12 (wm_yr_wk, wday, month, year, sell_price...)
   Категориални: 7 (item_id, dept_id, cat_id, store_id, state_id...)

🔧 Обработка на NaN стойности...
   ⚠️  sell_price: попълнено с медиана 3.9700

🔢 Scaling на числови данни...
   ✅ MinMaxScaler приложен (fit само върху train)
   Обхват на скалираните данни: [0.000, 1.000]

🏷️  Encoding на категориални данни...
   ✅ item_id: 565 категории
   ✅ dept_id: 2 категории
   ✅ cat_id: 1 категории
   ✅ store_id: 1 категории
   ✅ sta

## **8. Save pipeline objects**

**Цел:** Да може да се използва preprocessing при inference.

**Какво се запазва:**

- scaler.pkl
- label_encoders.pkl
- feature_list.pkl


In [9]:
# Проверка дали preprocessing обектите съществуват
try:
    scaler
    label_encoders
    numeric_cols
    categorical_cols
    feature_cols = list(X_train.columns)
    print("✅ Използване на preprocessing обекти от секция 7")
except NameError:
    # Ако не съществуват, зареждаме от комбинирания файл (ако съществува)
    preprocessing_path = MODELS_DIR / "preprocessing_objects.pkl"
    if preprocessing_path.exists():
        with open(preprocessing_path, "rb") as f:
            preprocessing_objects = pickle.load(f)
        scaler = preprocessing_objects["scaler"]
        label_encoders = preprocessing_objects["label_encoders"]
        numeric_cols = preprocessing_objects["numeric_cols"]
        categorical_cols = preprocessing_objects["categorical_cols"]
        feature_cols = preprocessing_objects["feature_cols"]
        print(f"✅ Заредени preprocessing обекти от {preprocessing_path}")
    else:
        raise ValueError("Няма налични preprocessing обекти! Изпълнете секция 7 първо.")

# 1) Запазване на scaler
scaler_path = MODELS_DIR / "scaler.pkl"
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"💾 Scaler запазен: {scaler_path}")

# 2) Запазване на label encoders
label_encoders_path = MODELS_DIR / "label_encoders.pkl"
with open(label_encoders_path, "wb") as f:
    pickle.dump(label_encoders, f)
print(f"💾 Label encoders запазени: {label_encoders_path}")
print(f"   Общо {len(label_encoders)} encoder-а: {', '.join(label_encoders.keys())}")

# 3) Запазване на feature list и метаданни
feature_list = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "feature_cols": feature_cols,
    "all_features": feature_cols,  # за обратна съвместимост
}

feature_list_path = MODELS_DIR / "feature_list.pkl"
with open(feature_list_path, "wb") as f:
    pickle.dump(feature_list, f)
print(f"💾 Feature list запазен: {feature_list_path}")
print(f"   Общо {len(feature_cols)} features")
print(f"   - Числови: {len(numeric_cols)}")
print(f"   - Категориални: {len(categorical_cols)}")

# 4) Допълнително: Запазване на метаданни като JSON (за четимост)
import json

metadata = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "feature_cols": feature_cols,
    "num_features": len(feature_cols),
    "num_numeric": len(numeric_cols),
    "num_categorical": len(categorical_cols),
    "label_encoder_categories": {
        col: list(enc.classes_)[:10]  # Първите 10 категории за информация
        for col, enc in label_encoders.items()
    },
}

metadata_path = MODELS_DIR / "preprocessing_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n📄 Metadata запазена (JSON): {metadata_path}")

print(f"\n✅ Pipeline обекти запазени успешно!")
print(f"   Файлове в {MODELS_DIR}:")
print(f"   - scaler.pkl")
print(f"   - label_encoders.pkl")
print(f"   - feature_list.pkl")
print(f"   - preprocessing_metadata.json (optional)")

print(f"\n💡 За зареждане при inference:")
print(f"   import pickle")
print(f"   scaler = pickle.load(open('{MODELS_DIR}/scaler.pkl', 'rb'))")
print(f"   label_encoders = pickle.load(open('{MODELS_DIR}/label_encoders.pkl', 'rb'))")
print(f"   feature_list = pickle.load(open('{MODELS_DIR}/feature_list.pkl', 'rb'))")


✅ Използване на preprocessing обекти от секция 7
💾 Scaler запазен: /Users/filipapopova/source/repos/2526-12b-feedforwardneuralnetwork-hierarchical-demand-forecasting/models/scaler.pkl
💾 Label encoders запазени: /Users/filipapopova/source/repos/2526-12b-feedforwardneuralnetwork-hierarchical-demand-forecasting/models/label_encoders.pkl
   Общо 7 encoder-а: item_id, dept_id, cat_id, store_id, state_id, event_name_1, event_type_1
💾 Feature list запазен: /Users/filipapopova/source/repos/2526-12b-feedforwardneuralnetwork-hierarchical-demand-forecasting/models/feature_list.pkl
   Общо 22 features
   - Числови: 12
   - Категориални: 7

📄 Metadata запазена (JSON): /Users/filipapopova/source/repos/2526-12b-feedforwardneuralnetwork-hierarchical-demand-forecasting/models/preprocessing_metadata.json

✅ Pipeline обекти запазени успешно!
   Файлове в /Users/filipapopova/source/repos/2526-12b-feedforwardneuralnetwork-hierarchical-demand-forecasting/models:
   - scaler.pkl
   - label_encoders.pkl
   - 

### **9. Sanity checks**

**Цел:** Проверка, че всичко е наред.

**Какво проверяваме:**

✅ няма NaN

✅ нормализирани стойности

✅ коректни shapes

✅ няма data leakage


In [11]:
# Проверка дали данните съществуват
try:
    X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test
    scaler
    label_encoders
except NameError:
    raise ValueError("Няма налични данни! Изпълнете секции 6 и 7 първо.")

print("🔍 Sanity checks...\n")

# 1) Проверка за NaN
print("1️⃣  NaN стойности:")
nan_ok = True
if X_train_scaled.isna().sum().sum() > 0:
    print("   ❌ X_train_scaled има NaN")
    nan_ok = False
if X_val_scaled.isna().sum().sum() > 0:
    print("   ❌ X_val_scaled има NaN")
    nan_ok = False
if X_test_scaled.isna().sum().sum() > 0:
    print("   ❌ X_test_scaled има NaN")
    nan_ok = False
if nan_ok:
    print("   ✅ Няма NaN стойности")

# 2) Проверка за нормализация
print("\n2️⃣  Нормализация:")
try:
    numeric_cols
except NameError:
    feature_list_path = MODELS_DIR / "feature_list.pkl"
    if feature_list_path.exists():
        with open(feature_list_path, "rb") as f:
            numeric_cols = pickle.load(f)["numeric_cols"]
    else:
        numeric_cols = []

if numeric_cols:
    train_min = X_train_scaled[numeric_cols].min().min()
    train_max = X_train_scaled[numeric_cols].max().max()
    if 0 <= train_min and train_max <= 1:
        print(f"   ✅ Train данните: [{train_min:.3f}, {train_max:.3f}]")
    else:
        print(f"   ⚠️  Train данните: [{train_min:.3f}, {train_max:.3f}]")

# 3) Проверка за shapes
print("\n3️⃣  Shapes:")
shapes_ok = True
if len(X_train_scaled) != len(y_train):
    print(f"   ❌ Train: X={X_train_scaled.shape}, y={y_train.shape}")
    shapes_ok = False
else:
    print(f"   ✅ Train: X={X_train_scaled.shape}, y={y_train.shape}")

if len(X_val_scaled) != len(y_val):
    print(f"   ❌ Val: X={X_val_scaled.shape}, y={y_val.shape}")
    shapes_ok = False
else:
    print(f"   ✅ Val: X={X_val_scaled.shape}, y={y_val.shape}")

if len(X_test_scaled) != len(y_test):
    print(f"   ❌ Test: X={X_test_scaled.shape}, y={y_test.shape}")
    shapes_ok = False
else:
    print(f"   ✅ Test: X={X_test_scaled.shape}, y={y_test.shape}")

# 4) Проверка за data leakage
print("\n4️⃣  Data leakage:")
print("   ✅ Scaler fit-нат само върху train (проверено в секция 7)")
print("   ✅ Label encoders fit-нати само върху train (проверено в секция 7)")

# Финален резултат
print("\n" + "="*50)
if nan_ok and shapes_ok:
    print("✅ ВСИЧКИ CHECKS ПРЕМИНАХА!")
else:
    print("⚠️  НЯКОИ CHECKS НЕ ПРЕМИНАХА!")
print("="*50)


🔍 Sanity checks...

1️⃣  NaN стойности:
   ✅ Няма NaN стойности

2️⃣  Нормализация:
   ✅ Train данните: [0.000, 1.000]

3️⃣  Shapes:
   ✅ Train: X=(755970, 22), y=(755970,)
   ✅ Val: X=(162155, 22), y=(162155,)
   ✅ Test: X=(162155, 22), y=(162155,)

4️⃣  Data leakage:
   ✅ Scaler fit-нат само върху train (проверено в секция 7)
   ✅ Label encoders fit-нати само върху train (проверено в секция 7)

✅ ВСИЧКИ CHECKS ПРЕМИНАХА!
